# 02 — Text Preprocessing
**Local version of `databricks/01_text_preprocessing.py`**

Databricks runs this on Spark for scale. This notebook shows the same logic locally for development and testing.

In [ ]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)

df = pd.read_csv('../data/sample/startups_sample.csv')
print(f'Loaded: {df.shape}')

In [ ]:
STOPWORDS = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """Normalize startup description for NLP pipeline."""
    if not isinstance(text, str):
        return ''
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = [lemmatizer.lemmatize(t) for t in text.split() if t not in STOPWORDS and len(t) > 2]
    return ' '.join(tokens)

df['description_clean'] = df['description'].apply(clean_text)

# Show before/after
print('BEFORE:', df['description'].iloc[0])
print('AFTER :', df['description_clean'].iloc[0])

In [ ]:
# Quality check
df['clean_word_count'] = df['description_clean'].str.split().str.len()
df['original_word_count'] = df['description'].str.split().str.len()
df['compression_ratio'] = df['clean_word_count'] / df['original_word_count']

print(f'Avg compression: {df["compression_ratio"].mean():.1%} of original tokens kept')
print(f'Empty after cleaning: {(df["clean_word_count"] == 0).sum()}')
print(df[['name','original_word_count','clean_word_count','compression_ratio']].head(8))

In [ ]:
# Save processed data locally
df.to_csv('../data/sample/startups_preprocessed.csv', index=False)
print('Saved: data/sample/startups_preprocessed.csv')
print(f'Columns: {list(df.columns)}')